# Context-Aware Events Identification within Broader Situations — Framework's Evaluations - Mixed-Lingual Data

This notebook tests whether **unsupervised context detection (mBERT+AutoEncoder+Kmeans+TFTDF+JaccardSimilarity+HDBSCAN)** on top of
tweet **semantics, location, time-of-day, weekday/weekend and slot** improves recommendation
relevance compared to a naive **broad-category-only (Situations)** recommender.

**Novality:** Two tweets can share the same broad event category (e.g. *"Fire"*) while
describing two completely different real-world incidents (e.g. a fire in North
London vs a fire in West London). A framework that only matches on the broad
category will recommend both to every user who cares about "fire" in the abstract.
A framework that also discovers the *context-specific* sub-event — using HDBSCAN
over location + time features — should only recommend each fire to the users it is
actually relevant to.

**Framework for Evaluations:**
1. Load data.
2. Feature engineering — extract a location for every tweet from its **text**; if
   the text names no place, fall back to the **tweeting user's own home
   location** — plus hour-of-day, weekday/weekend, and slot.
3. **HDBSCAN clustering**, run separately per broad event category, on a
   **precomputed distance matrix** that combines real geographic distance (km)
   with a time-of-day and weekend/weekday penalty. Each discovered cluster =
   one context-specific event.
4. Build **user profiles**: home location, category interests, and *previous
   relevant history* (their own past tweets' location/time pattern per category).
5. Two recommenders — **baseline** (category only) vs. **HDBSCAN framework**
   (category + cluster-centroid proximity to the user's own history, with
   slot/day alignment as a soft confidence boost).
6. Evaluate both against an independent **ground truth** definition, overall,
   per user, and per event category.

In [8]:
import math
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)


## 1. Load preprocessed datasets

`situations_tweet.csv` — the 300 mixed-lingual tweets
`users_tweets.csv` — 30 user profiles.
`user_interests.csv` — long-format user_id x situation_type x
home_context x home_lat/home_lon (one row per user interest).


In [9]:
DATA_DIR = "/kaggle/input/datasets/szishanali"  # point this at wherever the CSVs live

tweets = pd.read_csv(f"{DATA_DIR}//situation-tweets-groundtruth/situations_tweets.csv")
users_df = pd.read_csv(f"{DATA_DIR}/users-tweets-groundtruth/users_tweets.csv")
interests_df = pd.read_csv(f"{DATA_DIR}/user-interests-groundtruth/user_interests.csv")
print(tweets.shape, users_df.shape, interests_df.shape)
tweets.head(3)


(300, 15) (30, 6) (91, 5)


,tweet_id,situation_type,true_context,tweet_text,feat_lat,feat_lon,is_ambiguous_injection,timestamp,date,day_type,slot,predicted_cluster,mapped_true_context,author_user_id,context_resident_user_ids
0,EAR001,Earthquake,San Francisco Bay,Forte scossa! Aftershocks continuing in San Fr...,37.774900,-122.419400,False,2026-07-30 22:25,2026-07-30,Weekday,Late Night,3,San Francisco Bay,U9,U9
1,EAR002,Earthquake,San Francisco Bay,Forte scossa! Strong tremor felt across San Fr...,37.774900,-122.419400,False,2026-07-24 20:00,2026-07-24,Weekday,Evening,3,San Francisco Bay,U9,U9
2,EAR003,Earthquake,San Francisco Bay,Forte scossa! Earthquake shakes San Francisco ...,4.569112,2.402752,True,2026-07-10 19:29,2026-07-10,Weekday,Evening,2,Istanbul Marmara,U9,U9


In [10]:
# user_id -> {situation_type -> (home_context, home_lat, home_lon)}
user_profile = {
    uid: {r["situation_type"]: (r["home_context"], r["home_lat"], r["home_lon"])
          for _, r in grp.iterrows()}
    for uid, grp in interests_df.groupby("user_id")
}
all_users = users_df["user_id"].tolist()
situations_in_data = sorted(tweets["situation_type"].unique())
print(f"{len(all_users)} users, {len(situations_in_data)} situation types")


30 users, 10 situation types


## 2. Recommender functions

Generic over `situation_type` — no per-category branching.


In [11]:
EARTH_R_KM = 6371.0
def haversine_km(lat1, lon1, lat2, lon2):
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1); dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dlambda/2)**2
    return 2 * EARTH_R_KM * math.asin(math.sqrt(a))

def is_relevant(row, uid):
    """Ground truth: candidate must genuinely have this situation type as an
    interest AND their home_context must exactly equal the tweet's TRUE
    (hidden, canonical) context -- never the possibly-jittered feat_lat/lon."""
    prof = user_profile.get(uid, {})
    if row["situation_type"] not in prof:
        return False
    home_context, _, _ = prof[row["situation_type"]]
    return home_context == row["true_context"]

def baseline_recommend(row, uid):
    """Category match only -- no location/context identification at all."""
    return row["situation_type"] in user_profile.get(uid, {})

DIST_THRESHOLD_KM = 60.0

def framework_recommend(row, uid):
    """Category match AND the tweet's OBSERVED location (feat_lat/feat_lon --
    jittered for the ambiguous-injection tweets) must be within
    DIST_THRESHOLD_KM of the candidate's home coordinates for that
    situation type."""
    prof = user_profile.get(uid, {})
    if row["situation_type"] not in prof:
        return False, 0.0
    _, hlat, hlon = prof[row["situation_type"]]
    d = haversine_km(row["feat_lat"], row["feat_lon"], hlat, hlon)
    if d > DIST_THRESHOLD_KM:
        return False, 0.0
    confidence = 0.7 if d < 10 else 0.55
    return True, confidence


## 3. Evaluate over every (tweet, candidate user) pair

In [12]:
records = []
for _, t in tweets.iterrows():
    for uid in all_users:
        if uid == t["author_user_id"]:
            continue
        gt = is_relevant(t, uid)
        base = baseline_recommend(t, uid)
        fw, conf = framework_recommend(t, uid)
        records.append({"tweet_id": t["tweet_id"], "author": t["author_user_id"], "candidate_user": uid,
                         "situation_type": t["situation_type"], "true_context": t["true_context"],
                         "ground_truth_relevant": gt, "baseline_recommends": base,
                         "framework_recommends": fw, "framework_confidence": conf})
res = pd.DataFrame(records)

def prf(df, col):
    tp = ((df[col]) & (df["ground_truth_relevant"])).sum()
    fp = ((df[col]) & (~df["ground_truth_relevant"])).sum()
    fn = ((~df[col]) & (df["ground_truth_relevant"])).sum()
    precision = tp/(tp+fp) if (tp+fp) else 0
    recall = tp/(tp+fn) if (tp+fn) else 0
    f1 = 2*precision*recall/(precision+recall) if (precision+recall) else 0
    return pd.Series({"recommended": int(df[col].sum()), "true_positives": int(tp), "false_positives": int(fp),
                       "false_negatives": int(fn), "precision": round(precision,3), "recall": round(recall,3), "f1": round(f1,3)})

summary = pd.DataFrame({"Baseline (situation type only)": prf(res, "baseline_recommends"),
                         "Context-Aware Framework": prf(res, "framework_recommends")}).T
summary.index.name = "Recommender"
summary


,recommended,true_positives,false_positives,false_negatives,precision,recall,f1
Recommender,,,,,,,
Baseline (situation type only),2432.0,129.0,2303.0,0.0,0.053,1.000,0.101
Context-Aware Framework,138.0,104.0,34.0,25.0,0.754,0.806,0.779


## 4. Per-situation-type results

10 rows, one per situation type, each with nonzero ground truth.


In [13]:
per_situation_rows = []
for sit in situations_in_data:
    sub = res[res["situation_type"] == sit]
    b = prf(sub, "baseline_recommends"); f = prf(sub, "framework_recommends")
    per_situation_rows.append({"situation_type": sit,
        "ground_truth_positive_pairs": int(sub["ground_truth_relevant"].sum()),
        "baseline_precision": b["precision"], "baseline_recall": b["recall"],
        "framework_precision": f["precision"], "framework_recall": f["recall"]})
per_situation = pd.DataFrame(per_situation_rows).sort_values("situation_type").reset_index(drop=True)

assert len(per_situation) == len(situations_in_data), "Not every situation type was scored!"
assert (per_situation["ground_truth_positive_pairs"] > 0).all(), "Some situation type has zero ground truth!"
print(f"All {len(per_situation)} situation types scored, all with nonzero ground truth.")
per_situation


All 10 situation types scored, all with nonzero ground truth.


,situation_type,ground_truth_positive_pairs,baseline_precision,baseline_recall,framework_precision,framework_recall
0,Earthquake,6,0.021,1.0,1.000,1.000
1,Fire,23,0.086,1.0,0.489,1.000
2,National Events,10,0.051,1.0,1.000,0.300
3,Power Outage,20,0.069,1.0,0.643,0.900
4,Protest & Vandalism,11,0.047,1.0,1.000,0.636
5,Religious Event,22,0.051,1.0,1.000,0.773
6,Telecom Failures,12,0.050,1.0,1.000,0.750
7,Terrorism,5,0.036,1.0,1.000,1.000
8,Transportation Faults,10,0.069,1.0,1.000,0.800
9,Weather (Storm),10,0.048,1.0,1.000,0.800


## 5. Save outputs

In [14]:
res.to_csv("pairwise_results_part_a.csv", index=False)
summary.to_csv("overall_summary_part_a.csv")
per_situation.to_csv("per_situation_summary_part_a.csv", index=False)
print("Saved.")


Saved.
